# ADL Homework 4 — Convolutional Neural Networks on CIFAR-10

**Applied Deep Learning, Spring 2026**

We implement the PyTorch CIFAR-10 tutorial CNN, train it for image classification, and (in later tasks) extend it with a deconvolutional decoder for reconstruction and latent-feature analysis.

## Hyperparameters (Task 1)

We follow the [PyTorch CIFAR-10 tutorial](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) and the Lecture 7 demo. Because computational resources are limited, we train on **6,000 images** (one tenth of the CIFAR-10 training set), as allowed by the assignment.

| Parameter | Value |
|-----------|-------|
| Architecture | Conv(3→6, k=5) → ReLU → MaxPool → Conv(6→16, k=5) → ReLU → MaxPool → FC(400→120) → ReLU → FC(120→84) → ReLU → FC(84→10) |
| Loss | Cross-entropy |
| Optimizer | SGD (lr = 0.001, momentum = 0.9) |
| Batch size | 64 |
| Epochs | 40 |
| Training subset | **6,000** images (random sample, seed 42) |
| Test set | 10,000 images (full CIFAR-10 test set) |
| Input normalization | `ToTensor` + normalize to [-1, 1] with mean/std (0.5, 0.5, 0.5) |

In [ ]:
%matplotlib inline

from typing import List, cast

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.axes import Axes
from matplotlib.figure import Figure

from cifar_cnn import (
    CIFAR10_CLASSES,
    DEFAULT_BATCH_SIZE,
    DEFAULT_EPOCHS,
    DEFAULT_TRAIN_SUBSET,
    Net,
    collect_predictions,
    denormalize,
    get_device,
    make_dataloaders,
    per_class_accuracy,
    set_seed,
    train_model,
)
from deconv_net import (
    DEFAULT_LAMBDA,
    DeconvNet,
    collect_reconstructions,
    load_encoder_from_net,
    train_deconv_model,
)

plt.rcParams.update({"figure.figsize": (8, 5), "font.size": 11})

SEED = 42
TRAIN_SUBSET = DEFAULT_TRAIN_SUBSET
EPOCHS = DEFAULT_EPOCHS
TASK2_EPOCHS = DEFAULT_EPOCHS
LAMBDA = DEFAULT_LAMBDA
BATCH_SIZE = DEFAULT_BATCH_SIZE

set_seed(SEED)
device = get_device()
print(f"Device: {device}")
print(f"Training on {TRAIN_SUBSET} images for {EPOCHS} epochs")
print(f"Task 2 lambda: {LAMBDA}")

## Task 1 — CIFAR-10 classification

We train the CNN below and report training/test accuracy, learning curves, sample test predictions, and per-class accuracy.

In [ ]:
trainloader, testloader = make_dataloaders(
    train_subset_size=TRAIN_SUBSET,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

model = Net().to(device)
history = train_model(
    model,
    trainloader,
    testloader,
    device,
    epochs=EPOCHS,
)
per_class = per_class_accuracy(model, testloader, device)

final_train_acc = 100 * history[-1].train_acc
final_test_acc = 100 * history[-1].test_acc
print(f"\nFinal train accuracy: {final_train_acc:.1f}%")
print(f"Final test accuracy:  {final_test_acc:.1f}%")

In [ ]:
print("Epoch | train loss | train acc (%) | test loss | test acc (%)")
print("-" * 58)
for row in history:
    print(
        f"{row.epoch:5d} | {row.train_loss:10.3f} | "
        f"{100 * row.train_acc:13.1f} | {row.test_loss:9.3f} | "
        f"{100 * row.test_acc:12.1f}"
    )

print("\nPer-class test accuracy:")
print("Class   | Accuracy (%)")
print("-" * 22)
for cls in CIFAR10_CLASSES:
    print(f"{cls:7s} | {per_class[cls]:11.1f}")

In [ ]:
epochs = [row.epoch for row in history]
train_acc = [100 * row.train_acc for row in history]
test_acc = [100 * row.test_acc for row in history]

fig, ax = plt.subplots(figsize=(7, 4))
figure = cast(Figure, fig)
axis = cast(Axes, ax)
axis.plot(epochs, train_acc, marker="o", label="Train")
axis.plot(epochs, test_acc, marker="o", label="Test")
axis.set_xlabel("Epoch")
axis.set_ylabel("Accuracy (%)")
axis.set_title("Task 1: CIFAR-10 classification accuracy")
axis.legend()
axis.grid(True, alpha=0.3)
figure.tight_layout()
plt.show()

In [ ]:
images, labels, preds = collect_predictions(model, testloader, device, max_images=10)

n = images.size(0)
ncols = 5
nrows = (n + ncols - 1) // ncols

fig, axes_raw = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 2.2 * nrows))
figure = cast(Figure, fig)
axes = cast(List[Axes], list(np.atleast_1d(axes_raw).ravel()))

for idx in range(n):
    img = denormalize(images[idx]).numpy().transpose(1, 2, 0)
    true_name = CIFAR10_CLASSES[labels[idx]]
    pred_name = CIFAR10_CLASSES[preds[idx]]
    color = "green" if labels[idx] == preds[idx] else "red"
    axes[idx].imshow(np.clip(img, 0, 1))
    axes[idx].set_title(f"true: {true_name}\npred: {pred_name}", color=color, fontsize=8)
    axes[idx].axis("off")

for idx in range(n, len(axes)):
    axes[idx].axis("off")

figure.suptitle("Test images with predicted labels", fontsize=11)
figure.tight_layout()
plt.show()

### Discussion (Task 1)

We trained on **6,000 images** (one tenth of CIFAR-10) for **40 epochs** on CPU. The model reached **46.6% train** and **44.4% test** accuracy. Learning was slow for the first ~10 epochs (test accuracy below 18%), then improved steadily through epoch 40. This is below the PyTorch tutorial result on the full 50,000-image training set (~55% in 2 epochs), which is expected with less data and fewer total optimization steps.

Per-class test accuracy was uneven: `dog` (59.0%), `truck` (54.1%), and `plane` (53.5%) were strongest, while `cat` (20.1%) and `bird` (35.5%) were weakest. This suggests the small CNN captures coarse shape cues for vehicles and some animals, but struggles with fine-grained classes. Overall, the pipeline matches the tutorial and the reduced dataset is the main limitation rather than the implementation.

## Task 2 — Deconvolutional model

We extend the Task 1 CNN with a decoder that reconstructs the input image using **max-unpooling** and **transposed convolutions**. The network is trained with

$$
\mathcal{L} = \mathcal{L}_{ce} + \lambda \mathcal{L}_{rec},
$$

where $\mathcal{L}_{rec} = \frac{1}{3} \sum_{i=1}^{3} \|\tilde{x}_i - x_i\|_F^2$ is the mean per-channel MSE over RGB. We set $\lambda = 0.1$: large enough to produce visible reconstructions, but small enough that classification accuracy remains close to the classifier-only model. The encoder and classifier are initialized from the trained Task 1 weights; the decoder is trained from scratch.

In [ ]:
deconv_model = DeconvNet().to(device)
load_encoder_from_net(deconv_model, model)

deconv_history = train_deconv_model(
    deconv_model,
    trainloader,
    testloader,
    device,
    lam=LAMBDA,
    epochs=TASK2_EPOCHS,
)

final_test_acc_t2 = 100 * deconv_history[-1].test_acc
print(f"\nFinal test accuracy (Task 2): {final_test_acc_t2:.1f}%")
print(f"Lambda: {LAMBDA}")

In [ ]:
epochs_t2 = [row.epoch for row in deconv_history]
train_acc_t2 = [100 * row.train_acc for row in deconv_history]
test_acc_t2 = [100 * row.test_acc for row in deconv_history]

fig, ax = plt.subplots(figsize=(7, 4))
figure = cast(Figure, fig)
axis = cast(Axes, ax)
axis.plot(epochs_t2, train_acc_t2, marker="o", label="Train")
axis.plot(epochs_t2, test_acc_t2, marker="o", label="Test")
axis.set_xlabel("Epoch")
axis.set_ylabel("Accuracy (%)")
axis.set_title(f"Task 2: classification accuracy (lambda={LAMBDA})")
axis.legend()
axis.grid(True, alpha=0.3)
figure.tight_layout()
plt.show()

In [ ]:
originals, recons = collect_reconstructions(deconv_model, testloader, device, max_images=3)
n = originals.size(0)

fig, axes_raw = plt.subplots(n, 2, figsize=(4, 2.2 * n))
figure = cast(Figure, fig)
axes = cast(List[Axes], list(np.atleast_1d(axes_raw).ravel()))

for i in range(n):
    orig_img = denormalize(originals[i]).numpy().transpose(1, 2, 0)
    recon_img = denormalize(recons[i]).numpy().transpose(1, 2, 0)

    axes[2 * i].imshow(np.clip(orig_img, 0, 1))
    axes[2 * i].set_title("Original", fontsize=9)
    axes[2 * i].axis("off")

    axes[2 * i + 1].imshow(np.clip(recon_img, 0, 1))
    axes[2 * i + 1].set_title("Reconstruction", fontsize=9)
    axes[2 * i + 1].axis("off")

figure.suptitle("Task 2: original vs reconstructed test images", fontsize=11)
figure.tight_layout()
plt.show()

### Discussion (Task 2)

With $\lambda = 0.1$ and encoder/classifier weights initialized from Task 1, joint training reached **66.4% train** and **48.8% test** accuracy after 40 epochs. Test accuracy **improved by 4.4 percentage points** over Task 1 (44.4% → 48.8%), with a peak of **50.1%** at epoch 35. So the reconstruction term did not hurt classification; continued training on the combined objective slightly improved it.

Test reconstruction loss decreased from **0.278 to 0.254**, indicating the decoder learned a coarse inverse mapping, but the visual reconstructions remain blurry. This is typical for a shallow decoder and per-channel MSE on $32 \times 32$ images. We chose $\lambda = 0.1$ because it gave visible reconstructions while keeping the classification loss dominant early in training.

A noticeable **train–test gap** opened during Task 2 (66.4% vs 48.8%), suggesting mild overfitting on the 6,000-image subset once the decoder was added. Smaller $\lambda$ would likely preserve accuracy but produce poorer reconstructions; larger $\lambda$ would push reconstruction quality at the cost of classification performance.